In [3]:
import numpy as np

np.random.seed(42)

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

def layer_norm(x, gamma, beta, eps=1e-6):
    mean = np.mean(x, axis=-1, keepdims=True)
    var = np.var(x, axis=-1, keepdims=True)
    x_norm = (x - mean) / np.sqrt(var + eps)
    return gamma * x_norm + beta

def multi_head_attention(Q, K, V, W_q, W_k, W_v, W_o, num_heads):
    batch_size, seq_len, d_model = Q.shape
    d_k = d_model // num_heads

    Q = Q @ W_q
    K = K @ W_k
    V = V @ W_v

    Q = Q.reshape(batch_size, seq_len, num_heads, d_k).transpose(0,2,1,3)
    K = K.reshape(batch_size, seq_len, num_heads, d_k).transpose(0,2,1,3)
    V = V.reshape(batch_size, seq_len, num_heads, d_k).transpose(0,2,1,3)

    scores = Q @ K.transpose(0,1,3,2) / np.sqrt(d_k)
    weights = softmax(scores, axis=-1)

    attention = weights @ V

    attention = attention.transpose(0,2,1,3).reshape(batch_size, seq_len, d_model)

    output = attention @ W_o

    return output

def feed_forward(x, W1, b1, W2, b2):
    hidden = x @ W1 + b1
    hidden = np.maximum(0, hidden)
    output = hidden @ W2 + b2
    return output

def encoder_block(x, W_q, W_k, W_v, W_o, W1, b1, W2, b2, gamma1, beta1, gamma2, beta2, num_heads):
    attn_out = multi_head_attention(x, x, x, W_q, W_k, W_v, W_o, num_heads)
    x = layer_norm(x + attn_out, gamma1, beta1)

    ff_out = feed_forward(x, W1, b1, W2, b2)
    x = layer_norm(x + ff_out, gamma2, beta2)

    return x


batch_size = 2
seq_len = 5
d_model = 16
num_heads = 4
d_ff = 64

x = np.random.randn(batch_size, seq_len, d_model)

W_q = np.random.randn(d_model, d_model)
W_k = np.random.randn(d_model, d_model)
W_v = np.random.randn(d_model, d_model)
W_o = np.random.randn(d_model, d_model)

W1 = np.random.randn(d_model, d_ff)
b1 = np.random.randn(d_ff)

W2 = np.random.randn(d_ff, d_model)
b2 = np.random.randn(d_model)

gamma1 = np.ones(d_model)
beta1 = np.zeros(d_model)

gamma2 = np.ones(d_model)
beta2 = np.zeros(d_model)

output = encoder_block(
    x,
    W_q, W_k, W_v, W_o,
    W1, b1, W2, b2,
    gamma1, beta1,
    gamma2, beta2,
    num_heads
)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print(output)

Input shape: (2, 5, 16)
Output shape: (2, 5, 16)
[[[-0.17941374  1.17503504 -1.4165163   1.07900642  1.16104187
    0.12935419 -0.40698631 -0.73742143 -1.29911164 -0.34330006
    1.36702383 -1.00085104  1.75762348 -1.21110416  0.19450839
   -0.26888854]
  [ 0.65427425  0.93735216 -0.75979316 -0.06103546  1.00352128
   -0.73295374 -0.05963837 -0.82989692 -1.64831869 -0.57114897
    1.77195965 -1.28199865  1.49319555 -1.04786523  0.76671938
    0.36562691]
  [-0.98975214  0.95105738 -1.35337063 -0.12711396  0.2353188
    1.09665826 -1.61590246 -0.48594298 -0.74663269 -0.57768811
    1.59809942 -0.89724802 -0.32111669  1.17580888  0.63542104
    1.4224039 ]
  [ 2.33415954  0.15469283 -0.52486453 -0.56622811 -0.47823357
   -1.22179363 -1.32133396 -0.01492197 -0.55067954 -0.12483595
   -1.52743454  1.38253633  0.18103866  0.56489063  0.65442729
    1.05858054]
  [ 1.55833847  0.41502397 -0.58357969  0.16758131  1.55648423
   -1.89627884  0.24553555  0.4685427  -0.63123101  0.01757316
   -0.